# Week 1: Bonus Challenge - Polynomial Features

**Level:** Advanced/Optional  
**Time:** 30-45 minutes  
**Prerequisites:** Completed Week 1 post-class exercise

---

## 🎯 Learning Objectives

By completing this bonus challenge, you will:
1. Understand when linear regression fails (nonlinear relationships)
2. Learn how to add polynomial features to capture nonlinear patterns
3. Compare simple vs. polynomial regression performance
4. Preview feature engineering (covered in depth in Week 4)
5. See how model complexity affects performance

---

## 📚 Background: Why Polynomial Features?

**Problem:** Linear regression assumes relationships are... linear! But many real-world relationships are curved.

**Example:**
- Crop yield vs. fertilizer: Too little = low yield, too much = toxicity (curved relationship)
- House price vs. square feet: Diminishing returns at very large sizes (curved)

**Solution:** Add polynomial features!

**Simple linear regression:**
```
y = β₀ + β₁X
```

**Polynomial regression (degree 2):**
```
y = β₀ + β₁X + β₂X²
```

**This allows fitting curved relationships while still using LinearRegression!**

---

## 🚀 The Challenge

You will:
1. Create synthetic data with a **nonlinear** (quadratic) relationship
2. Fit simple linear regression (will perform poorly!)
3. Add polynomial features and refit
4. Compare performance
5. Apply to Diabetes dataset

---

**Let's begin!**

---

## Part 1: Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import load_diabetes

%matplotlib inline

print("✅ Imports successful!")

---

## Part 2: Create Synthetic Nonlinear Data

First, let's create data where the TRUE relationship is quadratic (y = X²), not linear.

In [ ]:
# Generate synthetic data with quadratic relationship
np.random.seed(42)

# Feature: X values from -3 to 3
X = np.linspace(-3, 3, 100).reshape(-1, 1)

# True relationship: y = 0.5*X^2 + noise
y_true = 0.5 * X.ravel() ** 2
y = y_true + np.random.normal(0, 0.5, 100)  # Add noise

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5, label='Data points')
plt.plot(X, y_true, color='green', linewidth=2, label='True relationship (y = 0.5X²)')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Synthetic Data with Quadratic (Nonlinear) Relationship')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: X = {X.shape}, y = {y.shape}")
print("\n⚠️ Notice the curved (parabolic) pattern - this is NOT linear!")

### 🔍 Observation

The green line shows the TRUE relationship: a parabola (y = 0.5X²).

**Question:** If we fit a straight line (simple linear regression) to this data, what will happen?

**Prediction:** The line will miss the curve! Let's test this.

---

## Part 3: Simple Linear Regression (Will Fail!)

Let's try fitting a simple linear model (y = β₀ + β₁X) to this curved data.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train simple linear regression
model_linear = LinearRegression()
model_linear.fit(X_train, y_train)

# Predict
y_pred_linear = model_linear.predict(X_test)

# Evaluate
r2_linear = r2_score(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))

print("=" * 60)
print("SIMPLE LINEAR REGRESSION RESULTS")
print("=" * 60)
print(f"R² Score:  {r2_linear:.4f}")
print(f"RMSE:      {rmse_linear:.4f}")
print("=" * 60)

# Visualize fit
X_plot = np.linspace(-3, 3, 100).reshape(-1, 1)
y_plot_linear = model_linear.predict(X_plot)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.6, label='Training data')
plt.scatter(X_test, y_test, alpha=0.6, color='orange', label='Test data')
plt.plot(X_plot, y_plot_linear, color='red', linewidth=2, label=f'Linear fit (R² = {r2_linear:.3f})')
plt.plot(X_plot, 0.5 * X_plot.ravel() ** 2, color='green', linestyle='--', linewidth=2, label='True relationship')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Simple Linear Regression on Nonlinear Data (Poor Fit!)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 📊 Analysis

**Observations:**
1. The red line (linear fit) is straight and misses the curve!
2. R² is probably very low or even negative (worse than predicting the mean)
3. The model systematically underestimates in the middle and overestimates at the edges

**This is a clear case where linear regression FAILS!**

---

## Part 4: Check Residual Plot (Diagnostic)

Let's create a residual plot to confirm the nonlinearity.

In [ ]:
# Calculate residuals
residuals_linear = y_test - y_pred_linear

# Residual plot
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_linear, residuals_linear, alpha=0.6, edgecolors='k', linewidth=0.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot for Simple Linear Regression')
plt.grid(True, alpha=0.3)

# Add annotation
plt.text(0.05, 0.95, '❌ CURVED PATTERN\nIndicates nonlinear relationship!',
         transform=plt.gca().transAxes,
         fontsize=12,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.show()

print("⚠️ The residual plot shows a clear CURVED pattern!")
print("This confirms that the relationship is nonlinear.")
print("\nWe need polynomial features to fix this!")

---

## Part 5: Add Polynomial Features (The Fix!)

Now let's add polynomial features to capture the curved relationship.

**PolynomialFeatures** transforms:
- [X] → [1, X, X²] (degree=2)

This allows our LINEAR model to fit a CURVED relationship!

In [ ]:
# Create polynomial features (degree 2: adds X²)
poly = PolynomialFeatures(degree=2, include_bias=False)

# Transform training and test data
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print("Original features:")
print(f"  X_train shape: {X_train.shape} (1 feature: X)")
print(f"\nAfter polynomial transformation:")
print(f"  X_train_poly shape: {X_train_poly.shape} (2 features: X, X²)")

# Show example transformation
print(f"\nExample transformation:")
print(f"  Original X: {X_train[0]}")
print(f"  Polynomial features: {X_train_poly[0]} → [X, X²]")

### 🔑 Key Understanding

**PolynomialFeatures** doesn't change the algorithm - we still use LinearRegression!

**It changes the features:**
- Before: 1 feature (X)
- After: 2 features (X, X²)

**Now the model can learn:**
```
y = β₀ + β₁X + β₂X²
```

This is still a LINEAR model (linear in the coefficients β), but it can fit a CURVED relationship!

---

## Part 6: Train Polynomial Regression

Now let's train LinearRegression using the polynomial features.

In [ ]:
# Train polynomial regression (LinearRegression with polynomial features)
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)

# Predict
y_pred_poly = model_poly.predict(X_test_poly)

# Evaluate
r2_poly = r2_score(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))

print("=" * 60)
print("POLYNOMIAL REGRESSION RESULTS (degree=2)")
print("=" * 60)
print(f"R² Score:  {r2_poly:.4f}")
print(f"RMSE:      {rmse_poly:.4f}")
print("=" * 60)

# Visualize fit
X_plot_poly = poly.transform(X_plot)
y_plot_poly = model_poly.predict(X_plot_poly)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.6, label='Training data')
plt.scatter(X_test, y_test, alpha=0.6, color='orange', label='Test data')
plt.plot(X_plot, y_plot_poly, color='blue', linewidth=2, label=f'Polynomial fit (R² = {r2_poly:.3f})')
plt.plot(X_plot, 0.5 * X_plot.ravel() ** 2, color='green', linestyle='--', linewidth=2, label='True relationship')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Polynomial Regression on Nonlinear Data (Much Better!)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n✅ The blue curve now matches the true green curve!")

### 🎉 Success!

**Observations:**
1. The blue curve (polynomial fit) closely follows the green curve (true relationship)!
2. R² should be much higher (close to 1.0)
3. RMSE should be much lower
4. The model captures the curved pattern

**This is the power of feature engineering!**

---

## Part 7: Check Residual Plot (Should Be Better!)

In [ ]:
# Calculate residuals
residuals_poly = y_test - y_pred_poly

# Residual plot
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_poly, residuals_poly, alpha=0.6, edgecolors='k', linewidth=0.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot for Polynomial Regression')
plt.grid(True, alpha=0.3)

# Add annotation
plt.text(0.05, 0.95, '✅ RANDOM SCATTER\nNo patterns - good fit!',
         transform=plt.gca().transAxes,
         fontsize=12,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.show()

print("✅ The residual plot now shows random scatter!")
print("This confirms that polynomial regression is appropriate for this data.")

---

## Part 8: Side-by-Side Comparison

In [ ]:
# Create comparison table
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(f"{'Metric':<25} {'Simple Linear':>20} {'Polynomial (degree=2)':>20}")
print("-" * 70)
print(f"{'R² Score':<25} {r2_linear:>20.4f} {r2_poly:>20.4f}")
print(f"{'RMSE':<25} {rmse_linear:>20.4f} {rmse_poly:>20.4f}")
print(f"{'Number of features':<25} {X_train.shape[1]:>20} {X_train_poly.shape[1]:>20}")
print("=" * 70)

improvement = ((r2_poly - r2_linear) / abs(r2_linear)) * 100 if r2_linear != 0 else float('inf')
print(f"\n📈 R² improvement: {improvement:.1f}%")
print(f"📉 RMSE reduction: {((rmse_linear - rmse_poly) / rmse_linear) * 100:.1f}%")

---

## Part 9: Apply to Diabetes Dataset

**Your turn!** Let's see if polynomial features help on a real dataset.

### Task 1: Load and Split Diabetes Data

In [ ]:
# TODO: Load Diabetes dataset
data = load_diabetes()
X = data.data
y = data.target

# TODO: Split data (80/20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Data loaded and split")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X_train.shape[1]}")

### Task 2: Baseline - Simple Linear Regression

In [ ]:
# TODO: Train simple linear regression
model_simple = LinearRegression()
model_simple.fit(X_train, y_train)

# TODO: Predict and evaluate
y_pred_simple = model_simple.predict(X_test)
r2_simple = r2_score(y_test, y_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, y_pred_simple))

print("=" * 60)
print("BASELINE: Simple Linear Regression")
print("=" * 60)
print(f"R² Score:  {r2_simple:.4f}")
print(f"RMSE:      {rmse_simple:.4f}")
print("=" * 60)

### Task 3: Add Polynomial Features (degree=2)

In [ ]:
# TODO: Create PolynomialFeatures transformer (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)

# TODO: Transform training data
X_train_poly = poly.fit_transform(X_train)

# TODO: Transform test data
X_test_poly = poly.transform(X_test)

print(f"Original features: {X_train.shape[1]}")
print(f"After polynomial transformation: {X_train_poly.shape[1]}")
print(f"\nNew features include: X₁, X₂, ..., X₁², X₁X₂, X₂², ...")

### Task 4: Train Polynomial Regression

In [ ]:
# TODO: Train model on polynomial features
model_poly_diabetes = LinearRegression()
model_poly_diabetes.fit(X_train_poly, y_train)

# TODO: Predict and evaluate
y_pred_poly_diabetes = model_poly_diabetes.predict(X_test_poly)
r2_poly_diabetes = r2_score(y_test, y_pred_poly_diabetes)
rmse_poly_diabetes = np.sqrt(mean_squared_error(y_test, y_pred_poly_diabetes))

print("=" * 60)
print("POLYNOMIAL REGRESSION (degree=2)")
print("=" * 60)
print(f"R² Score:  {r2_poly_diabetes:.4f}")
print(f"RMSE:      {rmse_poly_diabetes:.4f}")
print("=" * 60)

### Task 5: Compare Results

In [ ]:
# Comparison
print("=" * 70)
print("DIABETES DATASET: MODEL COMPARISON")
print("=" * 70)
print(f"{'Metric':<25} {'Simple Linear':>20} {'Polynomial (degree=2)':>20}")
print("-" * 70)
print(f"{'R² Score':<25} {r2_simple:>20.4f} {r2_poly_diabetes:>20.4f}")
print(f"{'RMSE':<25} {rmse_simple:>20.4f} {rmse_poly_diabetes:>20.4f}")
print(f"{'Number of features':<25} {X_train.shape[1]:>20} {X_train_poly.shape[1]:>20}")
print("=" * 70)

if r2_poly_diabetes > r2_simple:
    improvement = ((r2_poly_diabetes - r2_simple) / r2_simple) * 100
    print(f"\n✅ Polynomial features IMPROVED performance by {improvement:.1f}%")
else:
    print(f"\n⚠️ Polynomial features did NOT improve performance (might be overfitting!)")
    print("This is actually expected for Diabetes dataset - the relationships are mostly linear.")

---

## Part 10: Reflection Questions

Answer these questions to deepen your understanding:

### Question 1: Feature Explosion

**Observe:** The Diabetes dataset went from 10 features to how many polynomial features?

**Calculate:** With 10 original features and degree=2, you get:
- Original 10 features
- 10 squared terms (X₁², X₂², ..., X₁₀²)
- Interaction terms (X₁X₂, X₁X₃, ...) = 10 choose 2 = 45 terms
- **Total:** 65 features!

**Question:** What happens if you have 100 original features and use degree=2?

**Answer:** _________________________________________________________

**Implication:** Polynomial features can cause a "curse of dimensionality" - too many features! This is why we need **regularization** (Week 4).

---

### Question 2: When Did Polynomial Features Help?

**Compare your results:**
- Synthetic quadratic data: Polynomial features helped? YES / NO
- Diabetes dataset: Polynomial features helped? YES / NO

**Why the difference?**

**Answer:** _________________________________________________________

_____________________________________________________________________

**Hint:** Check the residual plots! If simple linear regression shows curved patterns, polynomial features will help. If it shows random scatter, polynomial features are unnecessary.

---

### Question 3: Overfitting Risk

**Experiment:** Try degree=3, degree=5, degree=10 on the synthetic data.

**Observation:** What happens to:
- Training R²: _________
- Test R²: _________

**Conclusion:** Higher degree polynomials can _________ (overfit/underfit) the data.

---

### Question 4: Real-World Application

Think of a problem in your field where the relationship might be nonlinear.

**Problem:** _________________________________________________________

**Why nonlinear?** ___________________________________________________

_____________________________________________________________________

**Would polynomial features help?** __________________________________

---

## 🎓 Key Takeaways

### What You Learned

1. ✅ **Residual plots reveal nonlinearity** - curved patterns indicate linear regression is inadequate

2. ✅ **Polynomial features capture curves** - transforming X → [X, X²] allows fitting curved relationships

3. ✅ **Still using LinearRegression!** - Polynomial regression is just linear regression with engineered features

4. ✅ **Feature explosion** - Polynomial features create many new features (can lead to overfitting)

5. ✅ **Not always helpful** - Only improves performance when relationships are truly nonlinear

---

### Connection to Future Weeks

**Week 3 (Tree-based Models):**
- Decision trees handle nonlinearity AUTOMATICALLY (no need for polynomial features!)
- This is why trees are so popular

**Week 4 (Regularization):**
- Ridge/Lasso regression prevent overfitting when you have many features
- Essential when using polynomial features!

**Week 6 (Neural Networks):**
- Neural networks can learn complex nonlinear relationships without manual feature engineering
- But require much more data

---

## 🚀 Bonus Challenges (If You Want More!)

### Challenge 1: Degree Comparison

Compare degree=1, 2, 3, 4, 5 on the Diabetes dataset. Plot training R² vs. test R² for each degree. What do you observe?

### Challenge 2: Feature Interactions

Use `PolynomialFeatures(degree=2, interaction_only=True)` - this creates only interaction terms (X₁X₂) without squared terms. Does this help?

### Challenge 3: Visualize Specific Feature

For Diabetes dataset, create scatter plots of each individual feature vs. target. Which features look nonlinear? Do polynomial features help more for those specific features?

---

## ✅ Completion Checklist

Before you finish:

- [ ] Understood why simple linear regression fails on curved data
- [ ] Successfully added polynomial features using `PolynomialFeatures`
- [ ] Compared simple vs. polynomial regression on synthetic data
- [ ] Applied polynomial features to Diabetes dataset
- [ ] Understood when polynomial features help (vs. when they don't)
- [ ] Recognized feature explosion problem
- [ ] Answered reflection questions

---

## 🎉 Congratulations!

You've completed the bonus challenge and learned about:
- Feature engineering
- Polynomial regression
- Model selection based on data characteristics
- The importance of residual plots

**This is advanced material** - well beyond Week 1 requirements. You're ahead of the curve!

---

**Questions?** Post in the discussion forum or bring to office hours.

**Next:** Preview Week 2 materials on Logistic Regression (Classification)!

---

*Last Updated: 2025-11-26*